# 04 — Model Evaluation

Цель ноутбука: показать classification metrics, confusion matrix, ROC curve, PR curve и ranking metrics.

In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import pandas as pd
import plotly.express as px

ROOT = Path.cwd()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT))

DATA_SYNTHETIC = ROOT / "data" / "synthetic"
DATA_PROCESSED = ROOT / "data" / "processed"
REPORTS = ROOT / "reports"
FIGURES = REPORTS / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 160)

print("Project root:", ROOT)
print("Synthetic data:", DATA_SYNTHETIC)

In [ ]:
from sklearn.metrics import auc, confusion_matrix, precision_recall_curve, roc_curve

metrics_path = REPORTS / "model_metrics.json"
ranking_path = REPORTS / "ranking_metrics.json"
predictions_path = REPORTS / "test_predictions.csv"

with open(metrics_path, "r", encoding="utf-8") as file:
    model_metrics = json.load(file)

with open(ranking_path, "r", encoding="utf-8") as file:
    ranking_metrics = json.load(file)

predictions = pd.read_csv(predictions_path)

display(pd.DataFrame([model_metrics]))

In [ ]:
y_true = predictions["success_label"]
y_pred = predictions["prediction"]
y_score = predictions["success_probability"]

cm = confusion_matrix(y_true, y_pred)
cm_df = pd.DataFrame(cm, index=["actual_0", "actual_1"], columns=["pred_0", "pred_1"])

display(cm_df)

In [ ]:
fig = px.imshow(cm_df, text_auto=True, title="Confusion matrix")
fig.show()
fig.write_html(FIGURES / "evaluation_confusion_matrix.html")

In [ ]:
fpr, tpr, _ = roc_curve(y_true, y_score)
roc_auc = auc(fpr, tpr)
roc_df = pd.DataFrame({"fpr": fpr, "tpr": tpr})

fig = px.line(roc_df, x="fpr", y="tpr", title=f"ROC curve AUC={roc_auc:.4f}")
fig.show()
fig.write_html(FIGURES / "evaluation_roc_curve.html")

In [ ]:
precision, recall, _ = precision_recall_curve(y_true, y_score)
pr_df = pd.DataFrame({"precision": precision, "recall": recall})

fig = px.line(pr_df, x="recall", y="precision", title="Precision-Recall curve")
fig.show()
fig.write_html(FIGURES / "evaluation_pr_curve.html")

In [ ]:
ranking_df = pd.DataFrame(ranking_metrics).T.reset_index().rename(columns={"index": "model"})
display(ranking_df)

fig = px.bar(
    ranking_df,
    x="model",
    y=["precision_at_1", "precision_at_3", "ndcg_at_3", "mrr"],
    barmode="group",
    title="Ranking metrics comparison",
)
fig.show()
fig.write_html(FIGURES / "evaluation_ranking_metrics.html")